# 7. Sentiment Classification with a Pretrained Transformer
In this tutorial you will use a **pretrained BERT-style model** to classify movie reviews as
**positive** or **negative** — without training the transformer itself from scratch.

This is the NLP equivalent of the *transfer learning* you did with ResNet for images in notebook 05:

| Computer Vision (NB 05) | NLP (this notebook) |
|---|---|
| Pretrained ResNet18 (ImageNet) | Pretrained DistilBERT (Wikipedia + books) |
| Freeze convolutional backbone | Freeze transformer layers |
| Train a new linear classifier head | Train a new linear classifier head |
| Classify cats vs dogs | Classify positive vs negative reviews |

Note: There are only **two small tasks**. Everything else is provided, and each task has a fully
worked solution hidden behind a 💡 in case you get stuck.


## A quick orientation

### What is HuggingFace 🤗?

HuggingFace is a company *and* an open-source ecosystem that has become the de facto standard
for working with pretrained NLP models. The two libraries we use:

| Library | What it gives you |
|---|---|
| `transformers` | Pretrained model architectures (BERT, GPT-2, T5, …) and their tokenizers, all behind one consistent API. |
| `datasets`     | Standard NLP datasets (GLUE, SQuAD, IMDB, …) with a unified, lazy-loading API. |

### What is DistilBERT?

DistilBERT is a smaller, faster version of **BERT** (Bidirectional Encoder Representations
from Transformers). It is **40% smaller** and **60% faster** than BERT-base while keeping
**~97% of its performance** on most NLP tasks. That makes it practical to run on a laptop CPU.

Under the hood it is a stack of 6 transformer encoder blocks (BERT-base has 12) — exactly
the architecture you saw in the lecture. It was pretrained on Wikipedia and BookCorpus with
**masked-language-modelling**: predict a missing word from its context. That pretraining
gives it rich linguistic representations that we will reuse here.

### What is the `[CLS]` token?

BERT prepends a special `[CLS]` token to every input sequence. Its final hidden state is
used as a **summary representation of the whole sentence** — perfect for classification
tasks like ours.

```
[CLS]  this  movie  was  great  [SEP]
  │
  └─► final hidden state used as sentence embedding (768-dim vector)
```


## Setup

If `transformers` and `datasets` are not installed yet, uncomment the install line and run the cell.


In [1]:
# !pip install transformers datasets --quiet

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

from transformers import DistilBertTokenizer, DistilBertModel
from datasets import load_dataset

torch.manual_seed(0)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


Device: cpu


## Step 1 — Load the SST-2 dataset

**SST-2** (Stanford Sentiment Treebank) contains short movie-review sentences labelled
positive (`1`) or negative (`0`). It is part of the **GLUE benchmark** — a standard
collection of NLP tasks.

The full training set has ~67k sentences. To keep training fast on CPU we use only a small subset.


In [9]:
raw = load_dataset("glue", "sst2")

# Use small, balanced subsets so this runs on a CPU in a few minutes
train_data = raw["train"].shuffle(seed=42).select(range(800))
val_data   = raw["validation"].shuffle(seed=42).select(range(200))

print(f"Train: {len(train_data)} examples | Validation: {len(val_data)} examples\n")

# Have a look at a few examples
for example in train_data.select(range(5)):
    sentiment = "positive 😊" if example["label"] == 1 else "negative 😞"
    print(f"  [{sentiment}]  {example['sentence'].strip()}")


Train: 800 examples | Validation: 200 examples

  [positive 😊]  klein , charming in comedies like american pie and dead-on in election ,
  [positive 😊]  be fruitful
  [positive 😊]  soulful and
  [positive 😊]  the proud warrior that still lingers in the souls of these characters
  [negative 😞]  covered earlier and much better


## Step 2 — Tokenization

Transformers don't take raw text. They take a **sequence of integer IDs**, where each ID
refers to an entry in a fixed **vocabulary**.

A **tokenizer** does the conversion from raw text do IDs. DistilBERT uses a *WordPiece* tokenizer that:

1. Splits text into subword units (e.g. `"transformers"` → `["transform", "##ers"]`).
2. Adds the special tokens `[CLS]` (at the start) and `[SEP]` (at the end) of the sentence.
3. **Pads or truncates** all sequences to a uniform length so we can batch them.
4. Produces an `attention_mask` — a parallel sequence of 0s and 1s telling the model
   which positions are real tokens (`1`) and which are padding (`0`).

Let's run it on a single sentence and see exactly what comes out.


In [3]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

demo = "Transformers are really powerful!"
enc  = tokenizer(demo, return_tensors="pt")

print("Original :", demo)
print("Tokens   :", tokenizer.convert_ids_to_tokens(enc["input_ids"][0]))
print("Input IDs:", enc["input_ids"])
print("Att. mask:", enc["attention_mask"])


Original : Transformers are really powerful!
Tokens   : ['[CLS]', 'transformers', 'are', 'really', 'powerful', '!', '[SEP]']
Input IDs: tensor([[  101, 19081,  2024,  2428,  3928,   999,   102]])
Att. mask: tensor([[1, 1, 1, 1, 1, 1, 1]])


Notice the `[CLS]` and `[SEP]` tokens that the tokenizer added automatically — these are
**not** part of the original sentence. The model has learned during pretraining to expect them.


## Step 3 — Wrap the data in a PyTorch Dataset

To use the standard PyTorch training tools (`DataLoader`, batching, shuffling), we wrap the
HuggingFace dataset in a `torch.utils.data.Dataset`. For each example we:

- Look up the sentence and label.
- Run the tokenizer on the sentence with **fixed length 64** (pad shorter, truncate longer).
- Return a dictionary with three tensors: `input_ids`, `attention_mask`, `label`.

This whole class is provided. Just read it once so you know what each batch looks like.


In [4]:
class SST2Dataset(Dataset):
    def __init__(self, data, tokenizer, max_length=64):
        self.data       = data
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sentence = self.data[idx]["sentence"]
        label    = self.data[idx]["label"]

        encoded = self.tokenizer(
            sentence,
            max_length=self.max_length,
            padding="max_length",       # pad shorter sentences to max_length
            truncation=True,            # cut off longer sentences at max_length
            return_tensors="pt",
        )

        return {
            "input_ids":      encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "label":          torch.tensor(label, dtype=torch.long),
        }


train_dataset = SST2Dataset(train_data, tokenizer)
val_dataset   = SST2Dataset(val_data,   tokenizer)

train_loader  = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=32, shuffle=False)

# Inspect one batch to confirm the shapes
batch = next(iter(train_loader))
print("input_ids      :", batch["input_ids"].shape)        # (16, 64)
print("attention_mask :", batch["attention_mask"].shape)   # (16, 64)
print("label          :", batch["label"].shape)            # (16,)


input_ids      : torch.Size([16, 64])
attention_mask : torch.Size([16, 64])
label          : torch.Size([16])


## Step 4 — Build the classifier

We will combine pretrained DistilBERT with a small **trainable classification head**:

```
input tokens
   │
   ▼
┌─────────────────────────┐
│  DistilBERT  (FROZEN)   │   ← no gradient updates, no learning here
└─────────────────────────┘
   │   last_hidden_state[:, 0, :]   ← the [CLS] vector
   ▼                                  shape (batch_size, 768)
┌───────────────────────────────┐
│  Linear(768 → 2)  (TRAINABLE) │
└───────────────────────────────┘
   │
   ▼   prediction — shape (batch_size, 2)
```

Two things to notice:

1. **Why freeze the transformer?** DistilBERT has ~66M parameters. Training all of them on
   our tiny 800-example dataset would be slow and probably overfit. The pretrained
   representations are already good enough — we only need to learn how to *read* them
   for our specific task. This is exactly the "feature extraction" pattern from Notebook 05.

2. **Why use only the `[CLS]` vector?** BERT outputs one 768-dimensional vector per token.
   For sentence-level classification, the `[CLS]` vector is the standard choice — during
   pretraining BERT learned to collect sentence-level information there.

### Task 1 — Complete the forward pass 

The constructor and freezing logic are written for you. You just need to:

1. Get the `[CLS]` representation from DistilBERT's output.
2. Pass it through the trainable linear layer.

**Hints:**

- `self.bert(input_ids=..., attention_mask=...)` returns an object whose
  `.last_hidden_state` attribute has shape `(batch_size, seq_len, 768)`.
- The `[CLS]` token is always at position **0** along the sequence dimension.
  So `.last_hidden_state[:, 0, :]` has shape `(batch_size, 768)` — perfect.
- Then call `self.classifier(...)` on it.

<details>
<summary>💡 Show full solution</summary>

```python
def forward(self, input_ids, attention_mask):
    outputs  = self.bert(input_ids=input_ids, attention_mask=attention_mask)
    cls_repr = outputs.last_hidden_state[:, 0, :]    # (batch, 768)  — the [CLS] vector
    return self.classifier(cls_repr)                 # (batch, 2)    — class logits
```

That's the whole transfer-learning trick: keep the pretrained backbone, train one small
new layer on top.
</details>


In [5]:
class SentimentClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")

        # Freeze every parameter in DistilBERT  →  no gradients computed for them
        for param in self.bert.parameters():
            param.requires_grad = False

        # The only trainable layer in the whole model
        self.classifier = nn.Linear(768, num_classes)

    def forward(self, input_ids, attention_mask):
        # ── TASK 1: 2 lines of code ──────────────────────────────
        # 1) Get DistilBERT's output for these tokens.
        # 2) Take the [CLS] vector and pass it through self.classifier.
        ...


# Sanity check — does a forward pass work and produce the right shape?
model = SentimentClassifier().to(DEVICE)

ids    = batch["input_ids"].to(DEVICE)
mask   = batch["attention_mask"].to(DEVICE)
prediction = model(ids, mask)

assert prediction.shape == (len(ids), 2), f"Expected ({len(ids)}, 2), got {prediction.shape}"
print("✓ Forward pass works. Output shape:", prediction.shape)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters    : {total:>11,}")
print(f"Trainable parameters: {trainable:>11,}  ({100 * trainable / total:.3f} % of total)")


AttributeError: 'NoneType' object has no attribute 'shape'

Notice: only **~0.002%** of all parameters in the model are actually being trained.
That's the magic of transfer learning — we're standing on the shoulders of a model that
was pretrained for thousands of hours on billions of words.


## Step 5 — Train the model

Standard PyTorch training loop: forward pass, compute cross-entropy loss, backward, step.
We train for **3 epochs**, which is plenty for the tiny linear head to converge.

Expect about **3–5 minutes on CPU**.


In [ ]:
def run_epoch(model, loader, optimizer, criterion, train=True):
    model.train() if train else model.eval()
    total_loss, correct, n = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            ids    = batch["input_ids"].to(DEVICE)
            mask   = batch["attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            prediction = model(ids, mask)
            loss   = criterion(prediction, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            correct    += (prediction.argmax(1) == labels).sum().item()
            n          += len(labels)

    return total_loss / len(loader), correct / n


# We only optimise the classifier head — the BERT params are frozen anyway,
# but passing only the trainable params to the optimiser is cleaner.
optimizer = Adam(model.classifier.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>8} | {'Val Acc':>7}")
print("-" * 55)

history = {k: [] for k in ("train_loss", "val_loss", "train_acc", "val_acc")}

for epoch in range(1, 4):
    tr_loss, tr_acc = run_epoch(model, train_loader, optimizer, criterion, train=True)
    va_loss, va_acc = run_epoch(model, val_loader,   optimizer, criterion, train=False)

    for k, v in zip(history, [tr_loss, va_loss, tr_acc, va_acc]):
        history[k].append(v)

    print(f"{epoch:>5} | {tr_loss:>10.4f} | {tr_acc:>9.3f} | {va_loss:>8.4f} | {va_acc:>7.3f}")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
eps = range(1, 4)

ax1.plot(eps, history["train_loss"], marker="o", label="Train")
ax1.plot(eps, history["val_loss"],   marker="o", label="Val")
ax1.set(xlabel="Epoch", ylabel="Loss", title="Cross-Entropy Loss")
ax1.legend(); ax1.grid()

ax2.plot(eps, history["train_acc"], marker="o", label="Train")
ax2.plot(eps, history["val_acc"],   marker="o", label="Val")
ax2.set(xlabel="Epoch", ylabel="Accuracy", title="Accuracy", ylim=(0, 1))
ax2.legend(); ax2.grid()

plt.tight_layout(); plt.show()


You should see validation accuracy around **80–85%** — not bad considering we only
trained ~1500 parameters on 800 examples!


## Step 6 — Predict on your own sentences

Now the fun part — use your trained model on any sentence you can think of.

### Task 2 — Complete the prediction function

You need to do four things:

1. **Tokenize** the input sentence (same arguments as in the Dataset: `max_length=64`, `padding`, `truncation`, `return_tensors="pt"`).
2. **Move** the tensors to `DEVICE`.
3. **Run** the model to get prediction.
4. **Pick** the class with the highest score using `torch.argmax`.

**Hints:**

- Call `tokenizer(sentence, max_length=..., padding="max_length", truncation=True, return_tensors="pt")` exactly as in the Dataset.
- Use `.to(DEVICE)` on `input_ids` and `attention_mask`.
- `prediction.argmax(dim=1)` returns the predicted class index (0 or 1). `.item()` extracts it as a Python int.

<details>
<summary>💡 Show full solution</summary>

```python
def predict_sentiment(sentence, model, tokenizer, max_length=64):
    model.eval()
    with torch.no_grad():
        encoded = tokenizer(
            sentence,
            max_length=max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        input_ids      = encoded["input_ids"].to(DEVICE)
        attention_mask = encoded["attention_mask"].to(DEVICE)

        prediction = model(input_ids, attention_mask)
        pred   = prediction.argmax(dim=1).item()
        return "positive" if pred == 1 else "negative"
```
</details>


In [ ]:
def predict_sentiment(sentence, model, tokenizer, max_length=64):
    """Return 'positive' or 'negative' for a single sentence."""
    model.eval()
    with torch.no_grad():
        # ── TASK 2 ───────────────────────────────────────────────
        # 1) tokenize `sentence` (max_length, padding, truncation, return_tensors)
        # 2) move input_ids and attention_mask to DEVICE
        # 3) run the model and take argmax over class dim
        # 4) return "positive" if class == 1 else "negative"
        ...


# Test it — feel free to add your own sentences
test_sentences = [
    "This movie was absolutely fantastic, I loved every minute!",
    "I wasted two hours of my life on this dreadful film.",
    "The plot was hard to follow but the visuals were stunning.",
    "Definitely the worst film I've seen this year.",
    "An instant classic — heartfelt and beautifully shot.",
]

for s in test_sentences:
    label = predict_sentiment(s, model, tokenizer)
    print(f"  [{label:>8}]  {s}")


---
## Wrap-up

You just built a working sentiment classifier on top of a 66-million-parameter pretrained
transformer — by training only **1500 parameters**.

### What you used

- **HuggingFace `transformers`** — to load DistilBERT and its tokenizer with one line each.
- **HuggingFace `datasets`** — to load SST-2 with one line.
- **PyTorch** — for the Dataset, DataLoader, training loop, and the new classifier head.

### What to try next:

1. **More data.** Bump `range(800)` to `range(5000)` and retrain. Does accuracy improve?
2. **Stronger head.** Replace `nn.Linear(768, 2)` with a small MLP
   (`Linear → ReLU → Dropout → Linear`). Does it help, or just overfit?
3. **Real fine-tuning.** Unfreeze BERT (`for p in self.bert.parameters(): p.requires_grad = True`),
   reduce the learning rate to `2e-5`, and train for 1 epoch. This usually gives a
   noticeable accuracy boost — but is much slower on CPU.
4. **Different task.** Swap SST-2 for the AG News dataset (`load_dataset("ag_news")`) —
   4-class topic classification. You only need to change `num_classes=4` and rename
   the label field.
